# WaveletSurfaceNet — final geometry-optimized training (Colab → Google Drive)

Trains the mixed-base model with the **v1 recipe + the new geometry-quality block** (`LAM_GEO=0.5`, on by
default in `train.py`), which adds field-level proxies for the metrics where we were weak: floaters/Chamfer,
components, holes, and closed-solid F-score. Trains for **120 epochs** to match the Tori paper's budget
(`tori/demo_wavetori.py`), on the full ModelNet40 train set, **auto-resuming across Colab sessions** (120
epochs won't fit one session) and saving all checkpoints + renders to your Google Drive.

**Recommended first:** set `EPOCHS=8`, run it, then locally `MIXED_CKPT=...assets/waveshape_geo.pt python
compare/rescore_ours.py` + `gen_table.py` to check the geo block actually *helps* before committing to the
full 120-epoch run (the earlier v2 retrain that boosted similar terms made things worse — verify, don't assume).

Runtime → **GPU** (A100/L4 strongly preferred for 120 epochs; T4 works but is slow — lean on auto-resume).

## 1· GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2· Configuration

In [ ]:
REPO_URL = "https://github.com/OlegJakushkin/Points_as_supertoroids.git"
BRANCH   = "main"
DRIVE_WS = "/content/drive/MyDrive/wsn_geo"      # persistent workspace in Drive
LOCAL    = "/content/wsn"

OUT    = "waveshape_geo"                          # checkpoints -> assets/<OUT>.pt (best) + _latest.pt
EPOCHS = 120                                      # Tori-paper budget; auto-resume accumulates across sessions
RES    = 32
BATCH  = 24                                       # A100: 48 ; T4: 16-24. Lower on CUDA OOM.
# geometry-quality block is ON by default (LAM_GEO=0.5 in train.py). To ablate it: EXTRA = "--lam-geo 0"
EXTRA  = ""
print("config OK | training the geo-optimized model for", EPOCHS, "epochs")

## 3· Mount Drive

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_WS, exist_ok=True)
print("workspace:", DRIVE_WS)

## 4· Get the code

In [ ]:
import os, subprocess
if os.path.isdir(LOCAL):
    subprocess.run(["git", "-C", LOCAL, "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", LOCAL, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", LOCAL, "pull", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, LOCAL], check=True)
os.chdir(LOCAL)
geo = any("lam_geo" in l for l in open("train.py"))
print("repo ready |", "geo-loss present" if geo else "NO geo-loss -- push the geometry-block changes first!")

## 5· Redirect outputs to Drive (persist + auto-resume)
`assets/`, `data/`, `renders/` become symlinks into Drive, so checkpoints survive disconnects and each session
resumes from `assets/<OUT>_latest.pt`.

In [ ]:
import os, shutil
for sub in ["assets", "data", "renders"]:
    dst = f"{DRIVE_WS}/{sub}"; os.makedirs(dst, exist_ok=True)
    src = f"{LOCAL}/{sub}"; os.makedirs(os.path.dirname(src), exist_ok=True)
    if os.path.islink(src):
        os.unlink(src)
    elif os.path.isdir(src):
        for f in os.listdir(src):
            s, d = f"{src}/{f}", f"{dst}/{f}"
            if not os.path.exists(d):
                (shutil.copytree if os.path.isdir(s) else shutil.copy2)(s, d)
        shutil.rmtree(src)
    os.symlink(dst, src)
    print(f"{sub:8s} -> {dst}")

## 6· Install dependencies

In [ ]:
!pip install -q trimesh rtree scikit-image networkx scipy
import trimesh, torch
print("trimesh", trimesh.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 7· Build the training data (cached in Drive, one time)

In [ ]:
import os, glob, zipfile, urllib.request
import numpy as np, torch
from waveshape import eval3d as E
from waveshape.datasets import load_mesh_normalized

DENSE, CAP = 1536, 9843
if not os.path.isdir("data/ModelNet40"):
    z = "data/ModelNet40.zip"
    if not os.path.exists(z):
        print("downloading ModelNet40 (~2GB, one time)...", flush=True)
        urllib.request.urlretrieve("http://modelnet.cs.princeton.edu/ModelNet40.zip", z)
    print("extracting...", flush=True)
    with zipfile.ZipFile(z) as zf: zf.extractall("data")

if not os.path.exists("data/se_clouds.pt"):
    files = sorted(glob.glob("data/ModelNet40/*/train/*.off"))[:CAP]
    print(f"building cloud cache from {len(files)} meshes...", flush=True)
    Ps, Ns = [], []
    for i, p in enumerate(files):
        try:
            m = load_mesh_normalized(p, max_faces=200000)
            Pp, Nn = E.sample_cloud(m, n=DENSE, noise=0.0, seed=0)
            if np.isfinite(Pp).all() and np.isfinite(Nn).all():
                Ps.append(Pp.astype(np.float32)); Ns.append(Nn.astype(np.float32))
        except Exception:
            continue
        if (i + 1) % 1000 == 0: print(f"  {i+1}/{len(files)} ({len(Ps)} ok)", flush=True)
    torch.save({"P": torch.tensor(np.stack(Ps)), "N": torch.tensor(np.stack(Ns))}, "data/se_clouds.pt")
    print(f"cached {len(Ps)} clouds", flush=True)
else:
    print("data/se_clouds.pt already cached in Drive", flush=True)

## 8· Train (geometry-optimized, 120 epochs, auto-resume)
Re-running this cell **resumes** from `assets/<OUT>_latest.pt`, so run it once per Colab session until it reaches
`EPOCHS`. The best-by-val checkpoint is `assets/<OUT>.pt`; the latest (de-floatered) is `assets/<OUT>_latest.pt`.

In [ ]:
import os
os.environ["MPLBACKEND"] = "Agg"
latest = f"assets/{OUT}_latest.pt"
args = f"--base mixed --region --out {OUT} --res {RES} --batch {BATCH} --epochs {EPOCHS}"
if os.path.exists(latest):                       # resume across sessions toward the 120-epoch target
    args += f" --resume {latest}"
if EXTRA: args += f" {EXTRA}"
print("python train.py", args, flush=True)
!python train.py {args} 2>&1 | tee -a {DRIVE_WS}/train_{OUT}.log

## 9· Artifacts in Drive (download the best checkpoint to score locally)

In [ ]:
import glob, os
for p in sorted(glob.glob(f"{DRIVE_WS}/assets/{OUT}*.pt")):
    print(f"  {p}  ({os.path.getsize(p)/1e6:.1f} MB)")
print("\nlatest val render:")
imgs = sorted(glob.glob(f"{DRIVE_WS}/renders/*val*.png"))
from IPython.display import Image, display
if imgs: display(Image(imgs[-1]))
print("\nNext: download assets/" + OUT + ".pt, then locally:")
print("  MIXED_CKPT=assets/" + OUT + ".pt python compare/rescore_ours.py   # re-score ONLY ours on the 2468-shape cache")
print("  python compare/gen_table.py                                       # see if the geo block improved the metrics")